In [1]:
import pandas as pd

df = pd.read_csv("files/processed_posts_dedup.csv")
s = df.sample(n=200, random_state=42).copy()

# labeling sheet: id + text only (blind to country, author, location)
sheet = s[["id", "full_text"]].copy()
sheet["full_text"] = sheet["full_text"].str.replace(r"\\n", " ", regex=True)
sheet["stance"] = ""
sheet["criticizes_israeli_actions"] = ""
sheet["notes"] = ""
sheet.to_csv("files/labeling_sheet.csv", index=False)

# 40 random ids to re-label later (agreement check)
relabel_ids = sheet["id"].sample(n=40, random_state=7)
relabel_ids.to_csv("files/relabel_ids.csv", index=False)

print(sheet.shape, relabel_ids.shape, sheet["id"].nunique())
print(sheet.columns.tolist())

(200, 5) (40,) 200
['id', 'full_text', 'stance', 'criticizes_israeli_actions', 'notes']


In [2]:
lab = pd.read_csv("files/labeling_sheet.csv")
print(lab["stance"].value_counts(dropna=False))
print(lab["criticizes_israeli_actions"].value_counts(dropna=False))

stance
NaN    200
Name: count, dtype: int64
criticizes_israeli_actions
NaN    200
Name: count, dtype: int64


In [5]:
import pandas as pd
lab = pd.read_csv("files/labeling_sheet.csv", dtype=str).fillna("")
print(lab.shape)
print(lab.loc[0, "full_text"])
print((lab["stance"] == "").sum(), "still unlabeled")

(200, 5)
West Bank behavior
200 still unlabeled


In [6]:
import pandas as pd

PATH = "files/labeling_sheet.csv"
lab = pd.read_csv(PATH, dtype=str).fillna("")

todo = lab.index[(lab["stance"] == "") | (lab["criticizes_israeli_actions"] == "")]
i = int(todo[0]) if len(todo) else len(lab)

def ask(prompt, valid):
    while True:
        v = input(prompt).strip().upper()
        if v in valid | {"Q", "B"}:
            return v

TARGET = 100   # stop automatically after 100 labeled posts

while i < len(lab) and (lab["stance"] != "").sum() < TARGET:
    print("\n" + "=" * 60, flush=True)
    print(f"Post {i+1}/200   labeled so far: {(lab['stance'] != '').sum()}   (Q = save & quit, B = back)", flush=True)
    print("=" * 60, flush=True)
    print(lab.at[i, "full_text"], flush=True)
    print("-" * 60, flush=True)

    s = ask("stance S/C/I/U: ", {"S", "C", "I", "U"})
    if s == "Q": break
    if s == "B":
        i = max(i - 1, 0)
        lab.loc[i, ["stance", "criticizes_israeli_actions", "notes"]] = ""
        continue

    f = ask("criticizes Israeli actions Y/N: ", {"Y", "N"})
    if f == "Q": break
    if f == "B": continue

    lab.at[i, "stance"] = s
    lab.at[i, "criticizes_israeli_actions"] = f
    lab.at[i, "notes"] = input("note (Enter to skip): ").strip()
    lab.to_csv(PATH, index=False)
    i += 1

lab.to_csv(PATH, index=False)
print("saved:", (lab["stance"] != "").sum(), "labeled")


Post 1/200   labeled so far: 0   (Q = save & quit, B = back)
West Bank behavior
------------------------------------------------------------

Post 2/200   labeled so far: 1   (Q = save & quit, B = back)
The ongoing jihadist terrorism is unfortunately not what @edsheeran meant when he mentioned the so-called 'West Bank'. 👇🏻
------------------------------------------------------------

Post 3/200   labeled so far: 2   (Q = save & quit, B = back)
🚨 POTENTIAL FIRST TEST OF ISRAEL’S NEW DEATH-PENALTY LAW  The captured terrorist could become the first suspect prosecuted under Israel’s new death-penalty law for deadly terrorist attacks in Judea and Samaria.  The law, pushed heavily by National Security Minister Itamar Ben-Gvir and enacted earlier this year, provides for the death penalty for a West Bank resident convicted of intentionally killing someone in a terrorist attack, unless a military court finds special circumstances warranting life imprisonment instead.  If authorities formally c

In [19]:
import pandas as pd
lab = pd.read_csv("files/labeling_sheet.csv")
done = lab[lab["stance"].notna()]

print(len(done), "labeled")
print(done["stance"].value_counts())
print(done["criticizes_israeli_actions"].value_counts())
print(pd.crosstab(done["stance"], done["criticizes_israeli_actions"]))
print(done["notes"].notna().sum(), "posts with notes")

100 labeled
stance
S    40
I    28
U    19
C    13
Name: count, dtype: int64
criticizes_israeli_actions
Y    54
N    46
Name: count, dtype: int64
criticizes_israeli_actions   N   Y
stance                            
C                           12   1
I                           16  12
S                            1  39
U                           17   2
0 posts with notes


In [21]:
import pandas as pd

PATH = "files/relabel_sheet.csv"
TARGET = 40

lab = pd.read_csv(PATH, dtype=str).fillna("")

todo = lab.index[(lab["stance"] == "") | (lab["criticizes_israeli_actions"] == "")]
i = int(todo[0]) if len(todo) else len(lab)

def ask(prompt, valid):
    while True:
        v = input(prompt).strip().upper()
        if v in valid | {"Q", "B"}:
            return v

while i < len(lab) and (lab["stance"] != "").sum() < TARGET:
    print("\n" + "=" * 60, flush=True)
    print(f"Post {i+1}/40   labeled so far: {(lab['stance'] != '').sum()}   (Q = save & quit, B = back)", flush=True)
    print("=" * 60, flush=True)
    print(lab.at[i, "full_text"], flush=True)
    print("-" * 60, flush=True)

    s = ask("stance S/C/I/U: ", {"S", "C", "I", "U"})
    if s == "Q": break
    if s == "B":
        i = max(i - 1, 0)
        lab.loc[i, ["stance", "criticizes_israeli_actions", "notes"]] = ""
        continue

    f = ask("criticizes Israeli actions Y/N: ", {"Y", "N"})
    if f == "Q": break
    if f == "B": continue

    lab.at[i, "stance"] = s
    lab.at[i, "criticizes_israeli_actions"] = f
    lab.at[i, "notes"] = input("note (Enter to skip): ").strip()
    lab.to_csv(PATH, index=False)   # saved after every post
    i += 1

lab.to_csv(PATH, index=False)
print("saved:", (lab["stance"] != "").sum(), "labeled")


Post 1/40   labeled so far: 0   (Q = save & quit, B = back)
Australia reportedly withdrew from talks on a joint statement by 12 governments supporting restrictions on trade with Israeli settlements in the occupied West Bank, despite having participated in negotiations until shortly before the statement was issued.  According to diplomatic sources cited by the ABC, Australia had indicated it was open to joining the initiative but ultimately pulled out.  The Australian government said it was pursuing targeted measures instead, citing concerns that broad trade restrictions could harm Palestinian businesses, workers and livelihoods.
------------------------------------------------------------



Post 2/40   labeled so far: 1   (Q = save & quit, B = back)
Israel-Gaza morning briefing, Sept 21, 2026  Israel marked Yom Kippur on Sept 20 without major fighting on any front and, for the first time in over a decade, without a single hostage held in Gaza -- all were recovered by January. A senior IDF commander called Gaza "calm" but warned against complacency about renewed major operations. Troops stay deployed in Gaza, south Lebanon and southwest Syria; Kerem Shalom crossing closed for the holiday.  That calm was shadowed by a deadly West Bank attack hours before the holiday: a gunman from Bidu shot and killed Netanel Shukrun, a father of six, at a spring near Neve Tzuf, then was captured wounded at a Ramallah hospital. Netanyahu vowed no terrorist or accomplice is immune; Ben Gvir wants the death penalty applied. The IDF has surged to roughly 27 battalions in the West Bank for mass arrest raids and to demolish the attacker's home.  In Gaza, the IDF says it killed several Hamas ope

In [24]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

a = pd.read_csv("files/labeling_sheet.csv")[["id", "stance", "criticizes_israeli_actions"]]
b = pd.read_csv("files/relabel_sheet.csv")[["id", "stance", "criticizes_israeli_actions"]]
m = a.merge(b, on="id", suffixes=("_1", "_2")).dropna()

print(len(m), "posts compared")
for col in ["stance", "criticizes_israeli_actions"]:
    same = (m[f"{col}_1"] == m[f"{col}_2"]).mean()
    k = cohen_kappa_score(m[f"{col}_1"], m[f"{col}_2"])
    print(f"{col}: same label {same:.0%}, kappa {k:.2f}")
print(pd.crosstab(m["stance_1"], m["stance_2"]))

40 posts compared
stance: same label 72%, kappa 0.60
criticizes_israeli_actions: same label 70%, kappa 0.36
stance_2  C  I   S  U
stance_1             
C         4  1   0  0
I         3  7   2  2
S         1  1  15  0
U         0  0   1  3


In [26]:
import re
import pandas as pd

lab = pd.read_csv("files/labeling_sheet.csv")
ids = lab.loc[lab["stance"].notna(), "id"]

df = pd.read_csv("files/processed_posts_dedup.csv")
t = df[df["id"].isin(ids)][["id", "full_text"]].copy()

def clean(x):
    x = re.sub(r"https?://\S+", "URL", str(x))
    x = re.sub(r"@\w+", "@user", x)
    return x.replace("\\n", " ")

t["full_text"] = t["full_text"].apply(clean)
t = t.sample(frac=1, random_state=1)       # shuffle order
t.to_csv("files/claude_test_100.csv", index=False)
print(t.shape, list(t.columns))

(100, 2) ['id', 'full_text']


In [27]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

h = pd.read_csv("files/labeling_sheet.csv")[["id", "stance", "criticizes_israeli_actions"]].dropna()
c = pd.read_csv("files/claude_labels_100.csv")
m = h.merge(c, on="id", suffixes=("_you", "_claude"))
print(len(m), "posts compared")

# main measure: S vs not-S
y, cl = m["stance_you"].eq("S"), m["stance_claude"].eq("S")
print("S vs not-S: same", f"{(y == cl).mean():.0%}", " kappa", f"{cohen_kappa_score(y, cl):.2f}")
print("you S:", y.sum(), " claude S:", cl.sum(), " both S:", (y & cl).sum())

# all 4 labels
print("4-label: same", f"{(m.stance_you == m.stance_claude).mean():.0%}",
      " kappa", f"{cohen_kappa_score(m.stance_you, m.stance_claude):.2f}")
print(pd.crosstab(m["stance_you"], m["stance_claude"]))

# flag
print("flag kappa:", f"{cohen_kappa_score(m.criticizes_israeli_actions_you, m.criticizes_israeli_actions_claude):.2f}")

100 posts compared
S vs not-S: same 63%  kappa 0.11
you S: 40  claude S: 9  both S: 6
4-label: same 46%  kappa 0.26
stance_claude  C   I  S   U
stance_you                 
C              5   6  1   1
I              1  24  2   1
S              0  30  6   4
U              0   8  0  11
flag kappa: 0.28


In [28]:
import pandas as pd
h = pd.read_csv("files/labeling_sheet.csv")[["id", "stance"]].dropna()
c = pd.read_csv("files/claude_labels_100.csv")
m = h.merge(c, on="id", suffixes=("_you", "_claude"))
g = m[(m.stance_you == "S") & (m.stance_claude == "I")]
print(len(g), "posts: you S, Claude I")
print("of these, Claude marked borderline:", (g.borderline == "Y").sum())
print(list(g["id"]))

30 posts: you S, Claude I
of these, Claude marked borderline: 7
[2101503212175921225, 2101066394192203904, 2101621961457877333, 2100744719445659914, 2100385181152559174, 2101766685661470886, 2101735391208362462, 2101486344337854921, 2101011787164590507, 2101015625237926157, 2100976902781505762, 2101469635082121682, 2101716795509948693, 2100790745078911000, 2101891922373570943, 2101722791666143417, 2101423382248194362, 2101321995044684093, 2101358624874565898, 2100704707706597515, 2101669234006687796, 2100823835255652588, 2100577345736249515, 2101312583122706821, 2101498826700050565, 2101551172016316812, 2100701670724583485, 2101783770458451968, 2099394849602421125, 2101894346434117725]


In [29]:
import pandas as pd, numpy as np
from sklearn.metrics import cohen_kappa_score

def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return round(100 * (c - h), 1), round(100 * (c + h), 1)

lab = pd.read_csv("files/labeling_sheet.csv"); lab = lab[lab["stance"].notna()]
names = {"S": "Explicit support", "I": "No explicit stance",
         "C": "Explicit opposition", "U": "Unclear / unrelated"}
rows = []
for k, nm in names.items():
    n = int((lab["stance"] == k).sum()); lo, hi = wilson(n, len(lab))
    rows.append((nm, n, round(100 * n / len(lab), 1), lo, hi))
pd.DataFrame(rows, columns=["label", "posts", "pct", "ci_low", "ci_high"]) \
  .to_csv("data/stance_pilot_counts.csv", index=False)

def agree(a, b, name):
    return (name, len(a), round(100 * (a == b).mean()), round(cohen_kappa_score(a, b), 2))

cols = ["id", "stance", "criticizes_israeli_actions"]
re_ = pd.read_csv("files/relabel_sheet.csv")[cols]
cl = pd.read_csv("files/claude_labels_100.csv")[cols]
m1 = lab[cols].merge(re_, on="id", suffixes=("_1", "_2")).dropna()
m2 = lab[cols].merge(cl, on="id", suffixes=("_1", "_2")).dropna()

rel = [
    agree(m1.stance_1, m1.stance_2, "Same labeler, re-label: 4 labels"),
    agree(m1.stance_1 == "S", m1.stance_2 == "S", "Same labeler, re-label: support vs not"),
    agree(m1.criticizes_israeli_actions_1, m1.criticizes_israeli_actions_2, "Same labeler, re-label: criticism flag"),
    agree(m2.stance_1, m2.stance_2, "Two labelers: 4 labels"),
    agree(m2.stance_1 == "S", m2.stance_2 == "S", "Two labelers: support vs not"),
    agree(m2.criticizes_israeli_actions_1, m2.criticizes_israeli_actions_2, "Two labelers: criticism flag"),
]
pd.DataFrame(rel, columns=["check", "posts", "same_pct", "kappa"]) \
  .to_csv("data/stance_pilot_reliability.csv", index=False)
print(pd.read_csv("data/stance_pilot_counts.csv"))
print(pd.read_csv("data/stance_pilot_reliability.csv"))

                 label  posts   pct  ci_low  ci_high
0     Explicit support     40  40.0    30.9     49.8
1   No explicit stance     28  28.0    20.1     37.5
2  Explicit opposition     13  13.0     7.8     21.0
3  Unclear / unrelated     19  19.0    12.5     27.8
                                    check  posts  same_pct  kappa
0        Same labeler, re-label: 4 labels     40        72   0.60
1  Same labeler, re-label: support vs not     40        88   0.75
2  Same labeler, re-label: criticism flag     40        70   0.36
3                  Two labelers: 4 labels    100        46   0.26
4            Two labelers: support vs not    100        63   0.11
5            Two labelers: criticism flag    100        63   0.28
